# Exploratory Data Analysis Practical (ABA Unit 1, Practical 2)

**Aim:** conduct exploratory data analysis (EDA) on a business dataset to identify trends and patterns.

In [1]:
# Step 1. Load the data and understand its structure
import pandas as pd

url = ("https://raw.githubusercontent.com/Prathameshv07/ai-ds-notes/main/"
       "content/assets/data/sahyadri-mart-sales.csv")
df = pd.read_csv(url, parse_dates=["date"])

print("Rows, columns:", df.shape)
print(df.dtypes.to_string())

Rows, columns: (8772, 16)
date               datetime64[us]
day_of_week                   str
store_id                      str
city                          str
region                        str
store_type                    str
store_area_sqft             int64
is_festival                 int64
discount_pct                int64
display                     int64
ad_spend_inr                int64
rainfall_mm               float64
footfall                    int64
transactions                int64
units_sold                  int64
revenue_inr                 int64


In [2]:
# Step 2. Check data quality: missing values and duplicate rows
missing = df.isna().sum()
print("Columns with missing values:")
print(missing[missing > 0].to_string())
print("Stores with missing rainfall:", sorted(df.loc[df["rainfall_mm"].isna(), "store_id"].unique()))
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate store-days:", df.duplicated(subset=["date", "store_id"]).sum())

Columns with missing values:
rainfall_mm    93
Stores with missing rainfall: ['S03', 'S10']
Duplicate rows: 0
Duplicate store-days: 0


In [3]:
# Step 3. Univariate analysis: categorical columns (count each store once)
stores = df.drop_duplicates("store_id")
print(stores["store_type"].value_counts().to_string())
print()
print(stores["region"].value_counts().to_string())

store_type
Express        7
Supermarket    5

region
Mumbai-Konkan          3
Western Maharashtra    3
North Maharashtra      2
Marathwada             2
Vidarbha               2


In [4]:
# Step 4. Univariate analysis: numerical columns (revenue shown in Rs thousand)
df["revenue_k"] = df["revenue_inr"] / 1000          # revenue in Rs thousand
print(df[["footfall", "transactions", "revenue_k"]].describe().round(1).to_string())
print("\nRevenue mean vs median (Rs thousand):",
      round(df["revenue_inr"].mean() / 1000, 1), "vs", round(df["revenue_inr"].median() / 1000, 1))

       footfall  transactions  revenue_k
count    8772.0        8772.0     8772.0
mean     1261.5         805.3      444.0
std       577.7         377.3      336.3
min       398.0         251.0       79.0
25%       767.8         490.0      167.4
50%      1175.0         738.0      325.9
75%      1617.0        1028.0      633.4
max      4744.0        3157.0     3004.7

Revenue mean vs median (Rs thousand): 444.0 vs 325.9


In [5]:
# Step 5. Compare groups: store type
by_type = df.groupby("store_type").agg(
    avg_footfall=("footfall", "mean"),
    avg_revenue_k=("revenue_k", "mean"),
    total_revenue=("revenue_inr", "sum"),
    total_bills=("transactions", "sum"),
)
by_type["avg_bill_inr"] = by_type["total_revenue"] / by_type["total_bills"]
print(by_type[["avg_footfall", "avg_revenue_k", "avg_bill_inr"]].round(0).to_string())

             avg_footfall  avg_revenue_k  avg_bill_inr
store_type                                            
Express             905.0          213.0         369.0
Supermarket        1761.0          767.0         683.0


In [6]:
# Step 6. Trend: total monthly revenue of the chain (Rs lakh), year by year
monthly = (df.groupby([df["date"].dt.year.rename("year"), df["date"].dt.month.rename("month")])
             ["revenue_inr"].sum() / 1e5).unstack("year")
print(monthly.round(0).to_string())
print("\nYearly total (Rs crore):", (df.groupby(df["date"].dt.year)["revenue_inr"].sum() / 1e7).round(1).to_dict())

year     2024    2025
month                
1      1478.0  1572.0
2      1365.0  1418.0
3      1529.0  1754.0
4      1548.0  1586.0
5      1611.0  1730.0
6      1425.0  1552.0
7      1387.0  1490.0
8      1456.0  1655.0
9      1598.0  1611.0
10     1934.0  2068.0
11     1816.0  1869.0
12     1704.0  1796.0

Yearly total (Rs crore): {2024: 188.5, 2025: 201.0}


In [7]:
# Step 7. Weekly pattern: average footfall per store by day of the week
order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
print(df.groupby("day_of_week")["footfall"].mean().reindex(order).round(0).to_string())

day_of_week
Monday       1143.0
Tuesday      1130.0
Wednesday    1161.0
Thursday     1183.0
Friday       1280.0
Saturday     1438.0
Sunday       1499.0


In [8]:
# Step 8. Relationships between numerical columns
cols = ["ad_spend_inr", "rainfall_mm", "discount_pct", "footfall", "revenue_inr"]
print(df[cols].corr().round(2).to_string())
# Is ad spend really that strongly linked to revenue? Check inside each store.
within = df.groupby("store_id")[["ad_spend_inr", "revenue_inr"]].corr().xs("revenue_inr", level=1)
print("\nAd spend vs revenue, within each store: median r =",
      round(within["ad_spend_inr"].median(), 2))

              ad_spend_inr  rainfall_mm  discount_pct  footfall  revenue_inr
ad_spend_inr          1.00         0.00          0.02      0.59         0.61
rainfall_mm           0.00         1.00         -0.01     -0.05        -0.02
discount_pct          0.02        -0.01          1.00      0.14         0.09
footfall              0.59        -0.05          0.14      1.00         0.96
revenue_inr           0.61        -0.02          0.09      0.96         1.00

Ad spend vs revenue, within each store: median r = 0.27


In [9]:
# Step 9. Effect of festivals and of heavy rain on footfall (Mumbai supermarket, S01)
s01 = df[df["store_id"] == "S01"]
print("Festival vs normal days:")
print(s01.groupby("is_festival")["footfall"].mean().round(0).to_string())
monsoon = s01[s01["date"].dt.month.isin([6, 7, 8, 9])]
rain_band = pd.cut(monsoon["rainfall_mm"], [-0.1, 0, 20, 50, 500],
                   labels=["dry", "light", "moderate", "heavy"])
print("\nMonsoon footfall by rain:")
print(monsoon.groupby(rain_band, observed=True)["footfall"].agg(["count", "mean"]).round(0).to_string())

Festival vs normal days:
is_festival
0    2169.0
1    3084.0

Monsoon footfall by rain:
             count    mean
rainfall_mm               
dry             69  2217.0
light           93  2018.0
moderate        59  1952.0
heavy           23  1851.0


In [10]:
# Step 10. Outliers in one store's daily revenue (1.5 x IQR rule)
rev = s01["revenue_inr"]
q1, q3 = rev.quantile([0.25, 0.75])
iqr = q3 - q1
low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
out = s01[(rev < low) | (rev > high)]
print(f"Q1 = {q1:,.0f}  Q3 = {q3:,.0f}  IQR = {iqr:,.0f}")
print(f"Outlier limits: below {low:,.0f} or above {high:,.0f}")
print("Outlier days:", len(out), "(all high) | festival days among them:", int(out["is_festival"].sum()))
other = out[out["is_festival"] == 0]
print("Non-festival outliers by month:", other["date"].dt.month.value_counts().sort_index().to_dict())
print("Non-festival outliers by weekday:", other["day_of_week"].value_counts().to_dict())
print("\nTop 3 days:")
print(out.nlargest(3, "revenue_inr")[["date", "is_festival", "revenue_inr"]].to_string(index=False))

Q1 = 909,030  Q3 = 1,197,120  IQR = 288,090
Outlier limits: below 476,895 or above 1,629,255
Outlier days: 25 (all high) | festival days among them: 15
Non-festival outliers by month: {3: 1, 10: 3, 11: 5, 12: 1}
Non-festival outliers by weekday: {'Sunday': 6, 'Friday': 2, 'Wednesday': 1, 'Saturday': 1}

Top 3 days:
      date  is_festival  revenue_inr
2025-10-21            1      3004710
2024-10-31            1      2214400
2024-09-07            1      2189690
